In [3]:
"""
Mobile Price Prediction - Full ML Pipeline
==========================================
Author: ML Engineer
Dataset: all_mobile_data5.csv (Nepal Mobile Prices)
"""

import re
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
np.random.seed(42)

# ─────────────────────────────────────────────────────────
# 1. RAW DATA LOADING
# ─────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1: LOADING RAW DATA")
print("=" * 60)

df_raw = pd.read_csv('/content/all_mobile_data5.csv')
print(f"Raw shape: {df_raw.shape}")
print(df_raw.head(10))

# ─────────────────────────────────────────────────────────
# 2. DATA PARSING & PREPROCESSING
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: DATA PARSING & PREPROCESSING")
print("=" * 60)

def parse_price(price_str):
    """Extract first numeric price value in NPR."""
    if pd.isna(price_str):
        return np.nan
    s = str(price_str)
    # Remove commas then find numbers
    s = s.replace(',', '')
    nums = re.findall(r'\d{4,7}', s)  # 4-7 digit numbers = realistic prices
    if nums:
        vals = [int(n) for n in nums]
        # Filter realistic Nepal mobile prices: 5000 - 500000
        vals = [v for v in vals if 5000 <= v <= 500000]
        return vals[0] if vals else np.nan
    return np.nan

def parse_ram(text):
    """Extract RAM in GB."""
    if pd.isna(text):
        return np.nan
    s = str(text)
    # Patterns: 12+256, 12/256, 12GB+1TB, 8GB, 16GB
    match = re.search(r'(\d+)\s*(?:GB|gb)?\s*[+/]\s*(?:\d+\s*(?:GB|gb|TB|tb)?)', s)
    if match:
        return int(match.group(1))
    # Plain GB pattern
    match = re.search(r'(\d+)\s*(?:GB|gb)', s)
    if match:
        val = int(match.group(1))
        if val <= 32:  # RAM is typically ≤ 32GB
            return val
    # Check x/y or x+y patterns
    match = re.search(r'(\d+)[/+](\d+)', s)
    if match:
        a, b = int(match.group(1)), int(match.group(2))
        if a <= 32 and b > 32:
            return a
        if b <= 32 and a > 32:
            return b
        if a <= 32 and b <= 32:
            return a  # both small, take first as RAM
    return np.nan

def parse_storage(text):
    """Extract storage in GB."""
    if pd.isna(text):
        return np.nan
    s = str(text)
    # TB -> GB
    tb_match = re.search(r'(\d+)\s*TB', s, re.IGNORECASE)
    if tb_match:
        return int(tb_match.group(1)) * 1024

    # x+y or x/y pattern
    match = re.search(r'(\d+)\s*(?:GB|gb)?\s*[+/]\s*(\d+)\s*(?:GB|gb)?', s)
    if match:
        a, b = int(match.group(1)), int(match.group(2))
        # Storage is usually the larger value if pair, or if one >32
        if a > 32 and b > 32:
            return max(a, b)
        if a > 32:
            return a
        if b > 32:
            return b
    # Single GB
    nums = re.findall(r'(\d+)\s*(?:GB|gb)', s)
    for n in nums:
        v = int(n)
        if v > 32:
            return v
    return np.nan

def clean_brand(brand_str):
    """Normalize brand to one of: Samsung, Apple, Xiaomi, Other."""
    if pd.isna(brand_str):
        return 'Other'
    s = str(brand_str).lower()
    if 'samsung' in s:
        return 'Samsung'
    if 'apple' in s:
        return 'Apple'
    if 'xiaomi' in s or 'redmi' in s or 'poco' in s:
        return 'Xiaomi'
    return 'Other'

def clean_model_name(model_str):
    """Extract clean model series info."""
    if pd.isna(model_str):
        return ''
    return str(model_str).strip()

def is_5g(model_str, price_str):
    combined = str(model_str) + str(price_str)
    return int('5G' in combined.upper() or '5g' in combined)

def is_ultra(model_str):
    return int('ultra' in str(model_str).lower())

def is_pro(model_str):
    return int('pro' in str(model_str).lower())

def is_foldable(model_str):
    return int('fold' in str(model_str).lower() or 'flip' in str(model_str).lower())

# Parse each row
records = []
for _, row in df_raw.iterrows():
    model = clean_model_name(row['Model'])
    price_raw = row['Price']
    brand_raw = row['Brand']

    price = parse_price(price_raw)
    if np.isnan(price):
        # Try Price_clean column
        price = parse_price(str(row.get('Price_clean', '')))

    if np.isnan(price):
        continue  # Skip rows with no parseable price

    ram = parse_ram(model)
    if np.isnan(ram):
        ram = parse_ram(price_raw)

    storage = parse_storage(model)
    if np.isnan(storage):
        storage = parse_storage(price_raw)

    brand = clean_brand(brand_raw)
    flag_5g = is_5g(model, price_raw)
    flag_ultra = is_ultra(model)
    flag_pro = is_pro(model)
    flag_fold = is_foldable(model)

    records.append({
        'Model': model,
        'Brand': brand,
        'Price_NPR': price,
        'RAM_GB': ram,
        'Storage_GB': storage,
        'Is_5G': flag_5g,
        'Is_Ultra': flag_ultra,
        'Is_Pro': flag_pro,
        'Is_Foldable': flag_fold,
    })

df = pd.DataFrame(records)
print(f"Parsed records: {len(df)}")
print(df.dtypes)
print(df.describe())
print(f"\nMissing values:\n{df.isnull().sum()}")

# ─────────────────────────────────────────────────────────
# 3. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: FEATURE ENGINEERING")
print("=" * 60)

# Price tier (for EDA, not used as feature)
def price_tier(p):
    if p < 20000: return 'Budget'
    elif p < 60000: return 'Mid-Range'
    elif p < 120000: return 'Upper-Mid'
    else: return 'Flagship'

df['Price_Tier'] = df['Price_NPR'].apply(price_tier)

# RAM × Storage interaction
df['RAM_x_Storage'] = df['RAM_GB'].fillna(0) * df['Storage_GB'].fillna(0)

# Log of storage (diminishing returns)
df['Log_Storage'] = np.log1p(df['Storage_GB'].fillna(0))

# Premium score: weighted combo
df['Premium_Score'] = (
    df['Is_Ultra'] * 3 +
    df['Is_Pro'] * 2 +
    df['Is_Foldable'] * 4 +
    df['Is_5G'] * 1
)

# Brand encoding
brand_map = {'Apple': 3, 'Samsung': 2, 'Xiaomi': 1, 'Other': 0}
df['Brand_Encoded'] = df['Brand'].map(brand_map)

print("Feature columns created:")
feature_cols = [
    'RAM_GB', 'Storage_GB', 'Is_5G', 'Is_Ultra', 'Is_Pro',
    'Is_Foldable', 'RAM_x_Storage', 'Log_Storage', 'Premium_Score', 'Brand_Encoded'
]
print(feature_cols)
print(df[feature_cols + ['Price_NPR']].describe())

# ─────────────────────────────────────────────────────────
# 4. EDA — EXPLORATORY DATA ANALYSIS
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: EXPLORATORY DATA ANALYSIS")
print("=" * 60)

fig = plt.figure(figsize=(20, 22))
fig.suptitle('Mobile Price Prediction — EDA Dashboard', fontsize=18, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

palette = {'Apple': '#555555', 'Samsung': '#1A73E8', 'Xiaomi': '#FF6900', 'Other': '#A8A8A8'}
tier_palette = {'Budget': '#4CAF50', 'Mid-Range': '#2196F3', 'Upper-Mid': '#FF9800', 'Flagship': '#E91E63'}

# 1. Price distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df['Price_NPR'], bins=20, color='#1A73E8', edgecolor='white', alpha=0.85)
ax1.set_title('Price Distribution (NPR)', fontweight='bold')
ax1.set_xlabel('Price (NPR)')
ax1.set_ylabel('Count')
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

# 2. Avg price by brand
ax2 = fig.add_subplot(gs[0, 1])
brand_avg = df.groupby('Brand')['Price_NPR'].mean().sort_values(ascending=False)
colors = [palette.get(b, '#888') for b in brand_avg.index]
bars = ax2.bar(brand_avg.index, brand_avg.values, color=colors, edgecolor='white')
ax2.set_title('Avg Price by Brand', fontweight='bold')
ax2.set_ylabel('Avg Price (NPR)')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
for bar, val in zip(bars, brand_avg.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
             f'{val/1000:.0f}K', ha='center', va='bottom', fontsize=8)

# 3. Brand count
ax3 = fig.add_subplot(gs[0, 2])
brand_cnt = df['Brand'].value_counts()
ax3.pie(brand_cnt.values, labels=brand_cnt.index,
        colors=[palette.get(b, '#888') for b in brand_cnt.index],
        autopct='%1.0f%%', startangle=90, pctdistance=0.8)
ax3.set_title('Brand Distribution', fontweight='bold')

# 4. Price by tier
ax4 = fig.add_subplot(gs[1, 0])
tier_order = ['Budget', 'Mid-Range', 'Upper-Mid', 'Flagship']
tier_data = [df[df['Price_Tier'] == t]['Price_NPR'].values for t in tier_order]
bp = ax4.boxplot(tier_data, labels=tier_order, patch_artist=True)
for patch, t in zip(bp['boxes'], tier_order):
    patch.set_facecolor(tier_palette[t])
    patch.set_alpha(0.8)
ax4.set_title('Price Distribution by Tier', fontweight='bold')
ax4.set_ylabel('Price (NPR)')
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax4.tick_params(axis='x', rotation=15)

# 5. RAM vs Price scatter
ax5 = fig.add_subplot(gs[1, 1])
ram_df = df.dropna(subset=['RAM_GB'])
for brand, grp in ram_df.groupby('Brand'):
    ax5.scatter(grp['RAM_GB'], grp['Price_NPR'],
                label=brand, color=palette.get(brand, '#888'), alpha=0.7, s=60, edgecolors='white')
ax5.set_title('RAM vs Price', fontweight='bold')
ax5.set_xlabel('RAM (GB)')
ax5.set_ylabel('Price (NPR)')
ax5.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax5.legend(fontsize=7)

# 6. Storage vs Price scatter
ax6 = fig.add_subplot(gs[1, 2])
stor_df = df.dropna(subset=['Storage_GB'])
for brand, grp in stor_df.groupby('Brand'):
    ax6.scatter(grp['Storage_GB'], grp['Price_NPR'],
                label=brand, color=palette.get(brand, '#888'), alpha=0.7, s=60, edgecolors='white')
ax6.set_title('Storage vs Price', fontweight='bold')
ax6.set_xlabel('Storage (GB)')
ax6.set_ylabel('Price (NPR)')
ax6.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax6.legend(fontsize=7)

# 7. Premium features comparison
ax7 = fig.add_subplot(gs[2, 0])
features = ['Is_5G', 'Is_Ultra', 'Is_Pro', 'Is_Foldable']
feat_labels = ['5G', 'Ultra', 'Pro', 'Foldable']
avg_with = [df[df[f] == 1]['Price_NPR'].mean() for f in features]
avg_without = [df[df[f] == 0]['Price_NPR'].mean() for f in features]
x = np.arange(len(feat_labels))
w = 0.35
ax7.bar(x - w/2, [v/1000 for v in avg_with], w, label='Has feature', color='#E91E63', alpha=0.85)
ax7.bar(x + w/2, [v/1000 for v in avg_without], w, label='No feature', color='#90CAF9', alpha=0.85)
ax7.set_title('Avg Price: Feature vs No Feature', fontweight='bold')
ax7.set_xticks(x); ax7.set_xticklabels(feat_labels)
ax7.set_ylabel('Avg Price (K NPR)')
ax7.legend(fontsize=8)

# 8. Correlation heatmap
ax8 = fig.add_subplot(gs[2, 1])
num_cols = ['Price_NPR', 'RAM_GB', 'Storage_GB', 'Is_5G', 'Is_Ultra',
            'Is_Pro', 'Is_Foldable', 'Premium_Score', 'Brand_Encoded']
corr_df = df[num_cols].dropna()
corr = corr_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax8, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            linewidths=0.5, annot_kws={'size': 7}, cbar_kws={'shrink': 0.8})
ax8.set_title('Feature Correlation Heatmap', fontweight='bold')
ax8.tick_params(axis='x', rotation=45, labelsize=7)
ax8.tick_params(axis='y', rotation=0, labelsize=7)

# 9. Price by 5G
ax9 = fig.add_subplot(gs[2, 2])
groups_5g = [df[df['Is_5G'] == 0]['Price_NPR'].dropna(),
             df[df['Is_5G'] == 1]['Price_NPR'].dropna()]
bp2 = ax9.boxplot(groups_5g, labels=['Non-5G', '5G'], patch_artist=True)
bp2['boxes'][0].set_facecolor('#90CAF9')
bp2['boxes'][1].set_facecolor('#1565C0')
for b in bp2['boxes']:
    b.set_alpha(0.8)
ax9.set_title('Price: 5G vs Non-5G', fontweight='bold')
ax9.set_ylabel('Price (NPR)')
ax9.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

# 10. Price tier count
ax10 = fig.add_subplot(gs[3, 0])
tier_cnt = df['Price_Tier'].value_counts()
tier_cnt = tier_cnt.reindex(tier_order).fillna(0)
bar_colors = [tier_palette[t] for t in tier_order]
ax10.bar(tier_order, tier_cnt.values, color=bar_colors, edgecolor='white', alpha=0.85)
ax10.set_title('Count by Price Tier', fontweight='bold')
ax10.set_ylabel('Count')
for i, v in enumerate(tier_cnt.values):
    ax10.text(i, v + 0.3, str(int(v)), ha='center', fontsize=9)

# 11. RAM distribution
ax11 = fig.add_subplot(gs[3, 1])
ram_dist = df['RAM_GB'].dropna().value_counts().sort_index()
ax11.bar(ram_dist.index.astype(str), ram_dist.values, color='#7E57C2', edgecolor='white', alpha=0.85)
ax11.set_title('RAM Distribution', fontweight='bold')
ax11.set_xlabel('RAM (GB)')
ax11.set_ylabel('Count')

# 12. Storage distribution
ax12 = fig.add_subplot(gs[3, 2])
stor_dist = df['Storage_GB'].dropna().value_counts().sort_index()
ax12.bar(stor_dist.index.astype(str), stor_dist.values, color='#26A69A', edgecolor='white', alpha=0.85)
ax12.set_title('Storage Distribution', fontweight='bold')
ax12.set_xlabel('Storage (GB)')
ax12.set_ylabel('Count')

plt.savefig('/content/eda_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.close()
print("EDA dashboard saved → eda_dashboard.png")

# ─────────────────────────────────────────────────────────
# 5. MODEL TRAINING
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: MODEL TRAINING")
print("=" * 60)

# Prepare clean modelling DataFrame
model_df = df.dropna(subset=['Price_NPR']).copy()

# Impute RAM and Storage with median
model_df['RAM_GB'] = model_df['RAM_GB'].fillna(model_df['RAM_GB'].median())
model_df['Storage_GB'] = model_df['Storage_GB'].fillna(model_df['Storage_GB'].median())
model_df['RAM_x_Storage'] = model_df['RAM_GB'] * model_df['Storage_GB']
model_df['Log_Storage'] = np.log1p(model_df['Storage_GB'])

X = model_df[feature_cols]
y = model_df['Price_NPR']

print(f"Dataset for modelling: {X.shape[0]} samples × {X.shape[1]} features")
print(f"Price range: {y.min():,.0f} – {y.max():,.0f} NPR")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

# Scale features for linear models
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Candidate models
models = {
    'Ridge Regression':         Ridge(alpha=10),
    'Decision Tree':            DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest':            RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':        GradientBoostingRegressor(n_estimators=100, random_state=42),
}

def evaluate(name, model, X_tr, y_tr, X_te, y_te, scaled=False):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    mae  = mean_absolute_error(y_te, preds)
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    r2   = r2_score(y_te, preds)
    cv   = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2')
    print(f"\n{'─'*40}")
    print(f"  {name}")
    print(f"  MAE : {mae:>10,.0f} NPR")
    print(f"  RMSE: {rmse:>10,.0f} NPR")
    print(f"  R²  : {r2:>10.4f}")
    print(f"  CV R²: {cv.mean():.4f} ± {cv.std():.4f}")
    return {'model': model, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'CV_R2': cv.mean()}

results = {}
for name, mdl in models.items():
    if 'Ridge' in name or 'Lasso' in name:
        res = evaluate(name, mdl, X_train_sc, y_train, X_test_sc, y_test, scaled=True)
    else:
        res = evaluate(name, mdl, X_train, y_train, X_test, y_test)
    results[name] = res

# ─────────────────────────────────────────────────────────
# 6. HYPERPARAMETER TUNING
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: HYPERPARAMETER TUNING (Random Forest)")
print("=" * 60)

param_grid = {
    'n_estimators':  [50, 100, 200],
    'max_depth':     [None, 5, 10, 15],
    'min_samples_split': [2, 5],
    'min_samples_leaf':  [1, 2],
    'max_features':  ['sqrt', 'log2'],
}

rf_base = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(
    rf_base, param_grid,
    cv=5, scoring='r2',
    n_jobs=-1, verbose=0
)
grid_search.fit(X_train, y_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV R²: {grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
best_mae  = mean_absolute_error(y_test, y_pred_best)
best_rmse = np.sqrt(mean_squared_error(y_test, y_pred_best))
best_r2   = r2_score(y_test, y_pred_best)

print(f"\nTuned Random Forest on Test Set:")
print(f"  MAE : {best_mae:>10,.0f} NPR")
print(f"  RMSE: {best_rmse:>10,.0f} NPR")
print(f"  R²  : {best_r2:>10.4f}")

results['Tuned Random Forest'] = {
    'model': best_rf, 'MAE': best_mae, 'RMSE': best_rmse, 'R2': best_r2,
    'CV_R2': grid_search.best_score_
}

# ─────────────────────────────────────────────────────────
# 7. SELECT BEST MODEL
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: BEST MODEL SELECTION")
print("=" * 60)

# Exclude Ridge since it used scaled data (different pipeline)
comparable = {k: v for k, v in results.items() if 'Ridge' not in k}
best_name = max(comparable, key=lambda k: comparable[k]['R2'])
best_result = comparable[best_name]

print(f"🏆  Best model: {best_name}")
print(f"    R²  : {best_result['R2']:.4f}")
print(f"    MAE : {best_result['MAE']:,.0f} NPR")
print(f"    RMSE: {best_result['RMSE']:,.0f} NPR")

best_model = best_result['model']

# ─────────────────────────────────────────────────────────
# 8. MODEL EVALUATION PLOTS
# ─────────────────────────────────────────────────────────
print("\nGenerating model evaluation plots...")

fig2, axes = plt.subplots(2, 3, figsize=(18, 11))
fig2.suptitle(f'Model Evaluation — {best_name}', fontsize=15, fontweight='bold')

model_names_plot = list(comparable.keys())
r2_vals   = [comparable[m]['R2']   for m in model_names_plot]
mae_vals  = [comparable[m]['MAE']  for m in model_names_plot]
rmse_vals = [comparable[m]['RMSE'] for m in model_names_plot]

bar_clr = ['#E91E63' if m == best_name else '#90CAF9' for m in model_names_plot]

# R² comparison
ax = axes[0, 0]
bars = ax.bar(model_names_plot, r2_vals, color=bar_clr, edgecolor='white')
ax.set_title('R² Score Comparison', fontweight='bold')
ax.set_ylabel('R²')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=20, labelsize=8)
for bar, v in zip(bars, r2_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{v:.3f}', ha='center', fontsize=8)

# MAE comparison
ax = axes[0, 1]
bars = ax.bar(model_names_plot, [m/1000 for m in mae_vals], color=bar_clr, edgecolor='white')
ax.set_title('MAE Comparison (K NPR)', fontweight='bold')
ax.set_ylabel('MAE (K NPR)')
ax.tick_params(axis='x', rotation=20, labelsize=8)
for bar, v in zip(bars, mae_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{v/1000:.1f}K', ha='center', fontsize=8)

# Actual vs Predicted
ax = axes[0, 2]
if 'Ridge' in best_name:
    y_pred_plot = best_model.predict(X_test_sc)
else:
    y_pred_plot = best_model.predict(X_test)
ax.scatter(y_test, y_pred_plot, alpha=0.7, color='#1A73E8', edgecolors='white', s=70)
mn, mx = min(y_test.min(), y_pred_plot.min()), max(y_test.max(), y_pred_plot.max())
ax.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect')
ax.set_title('Actual vs Predicted', fontweight='bold')
ax.set_xlabel('Actual Price (NPR)')
ax.set_ylabel('Predicted Price (NPR)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax.legend()

# Residuals
ax = axes[1, 0]
residuals = y_test.values - y_pred_plot
ax.scatter(y_pred_plot, residuals, alpha=0.7, color='#FF6900', edgecolors='white', s=70)
ax.axhline(0, color='red', linestyle='--', lw=2)
ax.set_title('Residual Plot', fontweight='bold')
ax.set_xlabel('Predicted Price (NPR)')
ax.set_ylabel('Residuals (NPR)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

# Residual distribution
ax = axes[1, 1]
ax.hist(residuals, bins=20, color='#7E57C2', edgecolor='white', alpha=0.85)
ax.axvline(0, color='red', linestyle='--', lw=2)
ax.set_title('Residual Distribution', fontweight='bold')
ax.set_xlabel('Residual (NPR)')
ax.set_ylabel('Count')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

# Feature importance
ax = axes[1, 2]
if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
    fi.plot(kind='barh', ax=ax, color='#26A69A', edgecolor='white')
    ax.set_title('Feature Importances', fontweight='bold')
    ax.set_xlabel('Importance')
else:
    ax.text(0.5, 0.5, 'Feature importance\nnot available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Feature Importances', fontweight='bold')

plt.tight_layout()
plt.savefig('/content/model_evaluation.png', dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.close()
print("Model evaluation plot saved → model_evaluation.png")

# ─────────────────────────────────────────────────────────
# 9. SAVE BEST MODEL
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: SAVING BEST MODEL")
print("=" * 60)

model_bundle = {
    'model':        best_model,
    'scaler':       scaler,
    'feature_cols': feature_cols,
    'brand_map':    brand_map,
    'model_name':   best_name,
    'metrics': {
        'R2': best_r2, 'MAE': best_mae, 'RMSE': best_rmse
    }
}

joblib.dump(model_bundle, '/content/mobile_price_model_2.pkl')
print(f"Model bundle saved → mobile_price_model.pkl")
print(f"  Model: {best_name}")
print(f"  Features: {feature_cols}")

# ─────────────────────────────────────────────────────────
# 10. TEST ON NEW / RANDOM DATA
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 9: TESTING ON NEW DATA (Random & Custom Phones)")
print("=" * 60)

# Load the saved model bundle
bundle = joblib.load('/content/mobile_price_model_2.pkl')
loaded_model = bundle['model']
cols = bundle['feature_cols']

def predict_price(ram_gb, storage_gb, brand, is_5g, is_ultra, is_pro, is_foldable):
    brand_enc = bundle['brand_map'].get(brand, 0)
    ram_x_stor = ram_gb * storage_gb
    log_stor   = np.log1p(storage_gb)
    premium    = is_ultra * 3 + is_pro * 2 + is_foldable * 4 + is_5g * 1
    row = pd.DataFrame([{
        'RAM_GB': ram_gb, 'Storage_GB': storage_gb,
        'Is_5G': is_5g, 'Is_Ultra': is_ultra, 'Is_Pro': is_pro, 'Is_Foldable': is_foldable,
        'RAM_x_Storage': ram_x_stor, 'Log_Storage': log_stor,
        'Premium_Score': premium, 'Brand_Encoded': brand_enc
    }])[cols]
    return loaded_model.predict(row)[0]

# Custom test phones
test_phones = [
    {'name': 'Budget Android (4GB/64GB)',
     'params': (4, 64, 'Xiaomi', 0, 0, 0, 0)},
    {'name': 'Mid-Range Samsung 5G (8GB/256GB)',
     'params': (8, 256, 'Samsung', 1, 0, 0, 0)},
    {'name': 'Samsung Galaxy S Ultra (12GB/512GB, 5G, Ultra)',
     'params': (12, 512, 'Samsung', 1, 1, 0, 0)},
    {'name': 'Apple iPhone Pro (8GB/256GB, Pro)',
     'params': (8, 256, 'Apple', 1, 0, 1, 0)},
    {'name': 'Apple iPhone Pro Max (12GB/1TB, Ultra/Pro)',
     'params': (12, 1024, 'Apple', 1, 1, 1, 0)},
    {'name': 'Samsung Z Fold (12GB/512GB, Foldable)',
     'params': (12, 512, 'Samsung', 1, 0, 0, 1)},
    {'name': 'Xiaomi Note (6GB/128GB, 5G)',
     'params': (6, 128, 'Xiaomi', 1, 0, 0, 0)},
    {'name': 'Xiaomi 15 Ultra (16GB/512GB, Ultra)',
     'params': (16, 512, 'Xiaomi', 1, 1, 0, 0)},
]

print(f"\n{'Phone':<50} {'Predicted Price (NPR)':>22}")
print("─" * 74)
for p in test_phones:
    predicted = predict_price(*p['params'])
    tier = price_tier(predicted)
    print(f"  {p['name']:<48} NPR {predicted:>10,.0f}  [{tier}]")

# Random test data
print("\n--- 5 Random Phones from Test Set ---")
test_sample = X_test.sample(5, random_state=7)
actual_sample = y_test.loc[test_sample.index]
pred_sample = loaded_model.predict(test_sample)
model_names_col = model_df.loc[test_sample.index, 'Model'].values

print(f"\n{'Model':<35} {'Actual':>12} {'Predicted':>12} {'Error':>12}")
print("─" * 73)
for i, (idx, row) in enumerate(test_sample.iterrows()):
    actual = actual_sample.loc[idx]
    pred   = pred_sample[i]
    err    = abs(actual - pred)
    mname  = model_names_col[i][:33]
    print(f"  {mname:<33} {actual:>10,.0f}   {pred:>10,.0f}   {err:>10,.0f}")

print("\n✅ Pipeline complete!")
print(f"\nSummary — Best Model: {best_name}")
print(f"  R² Score : {best_r2:.4f}  ({best_r2*100:.1f}% variance explained)")
print(f"  MAE      : NPR {best_mae:,.0f}")
print(f"  RMSE     : NPR {best_rmse:,.0f}")

STEP 1: LOADING RAW DATA
Raw shape: (127, 4)
                  Model                   Price    Brand     Price_clean
0  Samsung Mobiles List          Price in Nepal  Samsung  Price in Nepal
1       Galaxy Z Fold 7  Rs. 244,999 (12+256GB)  Samsung    244.99912256
2       Galaxy Z Fold 6  Rs. 164,999 (12+256GB)  Samsung    164.99912256
3       Galaxy Z Fold 6  Rs. 179,999 (12+512GB)  Samsung    179.99912512
4       Galaxy Z Flip 7  Rs. 154,999 (12+256GB)  Samsung    154.99912256
5       Galaxy Z Flip 6   Rs. 99,999 (12+256GB)  Samsung     99.99912256
6      Galaxy S26 Ultra  Rs. 292,999 (12GB+1TB)  Samsung      292.999121
7      Galaxy S26 Ultra  Rs. 242,999 (12/512GB)  Samsung    242.99912512
8      Galaxy S26 Ultra  Rs. 212,999 (12/256GB)  Samsung    212.99912256
9       Galaxy S26 Plus  Rs. 182,999 (12/256GB)  Samsung    182.99912256

STEP 2: DATA PARSING & PREPROCESSING
Parsed records: 109
Model           object
Brand           object
Price_NPR        int64
RAM_GB         float64
St